## UrbanRide_05 — Silver Payments
**Method:** Batch transformation — Bronze → Silver  
**Input:** `urbanride.bronze.payments`  
**Output:** `urbanride.silver.payments`  
**Schedule:** Daily · runs after UrbanRide_04 (Silver Trips must exist first)

**Load strategy:**
- First run → full load
- Daily runs → incremental append
- Watermark: `_ingested_at` (Bronze) vs `_processed_at` (Silver)
- Target grain: one clean row per `payment_id`

**Transformations:**
- Deduplication on `payment_id`
- Cast `payment_timestamp` STRING → TIMESTAMP
- Orphan detection — left join to `silver.trips`, flag no matching trip
- Fare mismatch detection — payment amount vs trip fare_amount delta > 5%
- Validate `payment_status` and `payment_method`
- Partition by `payment_status`

**Note:** Silver Trips must be built before this notebook runs.

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    to_timestamp, abs as spark_abs, round
)

CATALOG = 'urbanride'
BRONZE  = f'{CATALOG}.bronze'
SILVER  = f'{CATALOG}.silver'

spark.sql(f'USE CATALOG {CATALOG}')

VALID_STATUSES = ['success', 'failed']
VALID_METHODS  = ['UPI', 'Credit Card', 'Wallet', 'Cash']

# Fare mismatch threshold — flag if payment differs from fare by > 5%
FARE_MISMATCH_THRESHOLD = 0.05

print(f'Catalog  : {CATALOG}')
print(f'Source   : {BRONZE}.payments')
print(f'Target   : {SILVER}.payments')
print(f'Requires : {SILVER}.trips (must exist)')
print('Config ready.')


Catalog  : urbanride
Source   : urbanride.bronze.payments
Target   : urbanride.silver.payments
Requires : urbanride.silver.trips (must exist)
Config ready.


In [0]:
print('Checking load type...')

# Verify Silver trips exists — payments join depends on it
if not spark.catalog.tableExists(f'{SILVER}.trips'):
    raise Exception(f'{SILVER}.trips does not exist. Run UrbanRide_04 first.')

table_exists = spark.catalog.tableExists(f'{SILVER}.payments')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Silver table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {SILVER}.payments'
    ).collect()[0][0]
    print(f'  Silver table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Silver table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading bronze.payments...')

df_bronze_full = spark.table(f'{BRONZE}.payments')

if LOAD_TYPE == 'full':
    df_bronze = df_bronze_full
    print(f'  Full load — reading all rows')
else:
    df_bronze = df_bronze_full.filter(col('_ingested_at') > last_run)
    print(f'  Incremental — reading rows ingested after {last_run}')

input_count = df_bronze.count()
print(f'  Rows to process : {input_count:,}')

if input_count == 0:
    print('  No new rows — Silver already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Profile before cleaning
print()
print('Bronze profile (before cleaning):')
print(f'  Duplicate payment_ids  : {input_count - df_bronze.dropDuplicates(["payment_id"]).count():,}')
print(f'  Invalid status         : {df_bronze.filter(~col("payment_status").isin(VALID_STATUSES)).count():,}')
print(f'  Invalid method         : {df_bronze.filter(~col("payment_method").isin(VALID_METHODS)).count():,}')
print(f'  Fraud card flagged     : {df_bronze.filter(col("is_fraud_card") == True).count():,}')
print()
print('Payment status distribution:')
df_bronze.groupBy('payment_status').count().show()
print('Payment method distribution:')
df_bronze.groupBy('payment_method').count().orderBy('count', ascending=False).show()


Reading bronze.payments...
  Full load — reading all rows
  Rows to process : 19,600,753

Bronze profile (before cleaning):
  Duplicate payment_ids  : 0
  Invalid status         : 0
  Invalid method         : 0
  Fraud card flagged     : 53,017

Payment status distribution:
+--------------+--------+
|payment_status|   count|
+--------------+--------+
|       success|19015283|
|        failed|  585470|
+--------------+--------+

Payment method distribution:
+--------------+--------+
|payment_method|   count|
+--------------+--------+
|           UPI|10807836|
|   Credit Card| 3905308|
|        Wallet| 2932303|
|          Cash| 1955306|
+--------------+--------+



In [0]:
print('Deduplicating on payment_id...')

df_deduped = df_bronze.dropDuplicates(['payment_id'])

removed = input_count - df_deduped.count()
print(f'  Rows before : {input_count:,}')
print(f'  Rows after  : {df_deduped.count():,}')
print(f'  Removed     : {removed:,} duplicate rows')


Deduplicating on payment_id...
  Rows before : 19,600,753
  Rows after  : 19,600,753
  Removed     : 0 duplicate rows


In [0]:
# print('Casting payment_timestamp...')

# df_typed = df_deduped.withColumn(
#     'payment_timestamp',
#     to_timestamp(col('payment_timestamp'), 'yyyy-MM-dd HH:mm:ss')
# )

# bad_ts = df_typed.filter(col('payment_timestamp').isNull()).count()
# print(f'  NULL timestamps after cast : {bad_ts:,}')

# if bad_ts > 0:
#     print('  WARNING — some timestamps could not be parsed')
#     df_typed.filter(col('payment_timestamp').isNull()) \
#         .select('payment_id','payment_timestamp').show(5)


In [0]:
# Orphan payment — payment references a trip_id that does not exist
# in silver.trips completed trips
# ~0.3% of payments are orphans — embedded as data quality issue
print('Detecting orphan payments...')

# Read only completed trips from Silver — payments only exist for completed trips
# Filter before join — reduces right side from 21M to 19.5M rows
df_trips = spark.table(f'{SILVER}.trips') \
    .filter(col('trip_status') == 'completed') \
    .select('trip_id', 'fare_amount', 'city', 'pickup_time')

print(f'  Completed trips in Silver : {df_trips.count():,}')  # going fprward we wont need this

# Left join — unmatched payments get NULL trip columns
df_joined = df_deduped.join(df_trips, on='trip_id', how='left')

# Flag orphans — payment has no matching trip
df_orphan = df_joined.withColumn(
    'is_orphan_payment',
    col('fare_amount').isNull()  # fare_amount comes from trips — NULL means no match
)



Detecting orphan payments...
  Completed trips in Silver : 19,542,216
  Orphan payments detected : 58,537


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-7338841820854426>, line 25
     23 orphan_count = df_orphan.filter(col('is_orphan_payment') == True).count()
     24 print(f'  Orphan payments detected : {orphan_count:,}')
---> 25 print(f'  Orphan rate              : {round(orphan_count/df_deduped.count()*100, 3)}%')

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/utils.py:308, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    305 if is_remote() and "PYSPARK_NO_NAMESPACE_SHARE" not in os.environ:
    306     from pyspark.sql.connect import functions
--> 308     return getattr(functions, f.__name__)(*args, **kwargs)
    309 else:
    310     return f(*args, **kwargs)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/functions/builtin.py:855, in round(col, scale)
    853 scale = _enum_to_value(scale)
    854 scale = lit(

In [0]:
orphan_count = df_orphan.filter(col('is_orphan_payment') == True).count()
print(f'  Orphan payments detected : {orphan_count:,}')
print(f'  Orphan rate              : {(orphan_count/df_deduped.count()*100)}%')

  Orphan payments detected : 58,537
  Orphan rate              : 0.2986466897470725%


In [0]:
# Fare mismatch — payment amount differs from trip fare_amount by > 5%
# ~0.2% of payments have this issue — embedded as data quality signal // remove 
# Only check non-orphan payments — orphans have no fare to compare
print('Detecting fare mismatches...')

df_fare = df_orphan.withColumn(
    'fare_delta_pct',
    when(
        col('is_orphan_payment') == False,
        round(
            spark_abs(col('amount') - col('fare_amount')) / col('fare_amount'),
            4
        )
    ).otherwise(None)
).withColumn(
    'is_fare_mismatch',
    when(
        (col('is_orphan_payment') == False) &
        (col('fare_delta_pct') > FARE_MISMATCH_THRESHOLD),
        True
    ).otherwise(False)
)

mismatch_count = df_fare.filter(col('is_fare_mismatch') == True).count()
print(f'  Fare mismatches detected : {mismatch_count:,}')
print(f'  Mismatch rate            : {mismatch_count/df_deduped.count()*100}%')

# Show sample mismatches
if mismatch_count > 0:
    print('  Sample fare mismatches:')
    df_fare.filter(col('is_fare_mismatch') == True) \
        .select('payment_id','trip_id','amount','fare_amount','fare_delta_pct') \
        .limit(5).show(truncate=False)


Detecting fare mismatches...
  Fare mismatches detected : 33,565
  Mismatch rate            : 0.17124342110734214%
  Sample fare mismatches:
+------------------------------------+------------------------------------+------+-----------+--------------+
|payment_id                          |trip_id                             |amount|fare_amount|fare_delta_pct|
+------------------------------------+------------------------------------+------+-----------+--------------+
|b2adcf77-1e11-4bc9-a031-e7cdd9135be2|a7dc77ba-1693-4465-97d0-27390fd8c238|101.65|111.05     |0.0846        |
|d175062f-ec49-4b8a-bbb7-64485a9f042d|dab4ff4c-2978-4882-8652-51c4d978966e|49.02 |55.28      |0.1132        |
|276749fc-27c0-43ba-868f-544387b9e0f0|e052dd05-fac8-488a-8501-77540e4266b7|290.73|249.58     |0.1649        |
|328ad26b-3f63-496d-8691-144929314d10|c0748dbd-85c0-4e20-bec5-1a425a540e4d|158.51|123.65     |0.2819        |
|534f53a2-2ae2-486d-b35f-d419da7f6c6e|477deb9d-2b8c-4b56-9631-3f0207fc7b05|82.91 |109.34 

In [0]:
print('Validating payment status and method...')

df_validated = df_fare \
    .withColumn('is_status_invalid',
        ~col('payment_status').isin(VALID_STATUSES)
    ) \
    .withColumn('is_method_invalid',
        ~col('payment_method').isin(VALID_METHODS)
    )

print(f'  Invalid status rows : {df_validated.filter(col("is_status_invalid") == True).count():,}')
print(f'  Invalid method rows : {df_validated.filter(col("is_method_invalid") == True).count():,}')


Validating payment status and method...
  Invalid status rows : 0
  Invalid method rows : 0


In [0]:
# Drop trip columns brought in by join — keep only payment columns + flags
# fare_amount from trips is kept as trip_fare_amount for reference
df_silver = df_validated \
    .withColumnRenamed('fare_amount', 'trip_fare_amount') \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit(f'{BRONZE}.payments'))

# Drop trip join columns not needed in Silver payments
cols_to_drop = ['city', 'pickup_time']
df_silver = df_silver.drop(*cols_to_drop)

print(f'Final row count   : {df_silver.count():,}')
print(f'Final column count: {len(df_silver.columns)}')
print()
df_silver.printSchema()


Final row count   : 19,600,753
Final column count: 18

root
 |-- trip_id: string (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_timestamp: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- card_hash: string (nullable = true)
 |-- is_fraud_card: boolean (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- trip_fare_amount: double (nullable = true)
 |-- is_orphan_payment: boolean (nullable = false)
 |-- fare_delta_pct: double (nullable = true)
 |-- is_fare_mismatch: boolean (nullable = false)
 |-- is_status_invalid: boolean (nullable = true)
 |-- is_method_invalid: boolean (nullable = true)
 |-- _processed_at: timestamp (nullable = false)
 |-- _source: string (nullable = false)



In [0]:
import time
print(f'Writing silver.payments — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_silver.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('payment_status') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{SILVER}.payments')
    print(f'  Full load written : {df_silver.count():,} rows')
    print(f'  Write time : {time.time()-t0}s')
    print('Running OPTIMIZE...')
    spark.sql(f'OPTIMIZE {SILVER}.payments')
    print('  OPTIMIZE complete')
else:
    new_count = df_silver.count()
    if new_count > 0:
        df_silver.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{SILVER}.payments')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time : {time.time()-t0}s')
        print('Running OPTIMIZE...')
        spark.sql(f'OPTIMIZE {SILVER}.payments')
        print('  OPTIMIZE complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing silver.payments — mode: full...
  Full load written : 19,600,753 rows
  Write time : 36.015328884124756s
Running OPTIMIZE...
  OPTIMIZE complete


In [0]:
print('=== SILVER PAYMENTS VERIFICATION ===')
print()

df_verify = spark.table(f'{SILVER}.payments')
total = df_verify.count()

print(f'  Total rows : {total:,}')
print(f'  Load type  : {LOAD_TYPE}')
print()

print('Data quality checks:')
checks = [
    ('Null payment_id',     df_verify.filter(col('payment_id').isNull()).count(),    False),
    ('Null timestamp',      df_verify.filter(col('payment_timestamp').isNull()).count(), False),
    ('Invalid status',      df_verify.filter(col('is_status_invalid') == True).count(), False),
    ('Invalid method',      df_verify.filter(col('is_method_invalid') == True).count(), False),
    ('Orphan payments',     df_verify.filter(col('is_orphan_payment') == True).count(),  True),
    ('Fare mismatches',     df_verify.filter(col('is_fare_mismatch') == True).count(),   True),
    ('Fraud card flagged',  df_verify.filter(col('is_fraud_card') == True).count(),      True),
]

for check, result, is_signal in checks:
    if is_signal:
        status = 'INFO'
    else:
        status = 'WARN' if result > 0 else 'PASS'
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Payment status distribution:')
df_verify.groupBy('payment_status').count().orderBy('count', ascending=False).show()

print('Payment method distribution:')
df_verify.groupBy('payment_method').count().orderBy('count', ascending=False).show()

print('Sample rows:')
df_verify.select(
    'payment_id','trip_id','amount','trip_fare_amount',
    'payment_method','payment_status',
    'is_orphan_payment','is_fare_mismatch','is_fraud_card'
).limit(5).show(truncate=False)

print()
print('Silver payments ready. Next: UrbanRide_06 — Silver Clickstream')


=== SILVER PAYMENTS VERIFICATION ===

  Total rows : 19,600,753
  Load type  : full

Data quality checks:
  PASS  Null payment_id                : 0
  PASS  Null timestamp                 : 0
  PASS  Invalid status                 : 0
  PASS  Invalid method                 : 0
  INFO  Orphan payments                : 58,537
  INFO  Fare mismatches                : 33,565
  INFO  Fraud card flagged             : 53,017

Payment status distribution:
+--------------+--------+
|payment_status|   count|
+--------------+--------+
|       success|19015283|
|        failed|  585470|
+--------------+--------+

Payment method distribution:
+--------------+--------+
|payment_method|   count|
+--------------+--------+
|           UPI|10807836|
|   Credit Card| 3905308|
|        Wallet| 2932303|
|          Cash| 1955306|
+--------------+--------+

Sample rows:
+------------------------------------+------------------------------------+------+----------------+--------------+--------------+-----------